In [ ]:
!pip -q install -U google-genai pydantic pandas

In [ ]:
import json
import re
import time
import pandas as pd

from typing import Optional, List
from pydantic import BaseModel
from google import genai

from google.colab import userdata, files

API_KEY = userdata.get("GEMINI_API_KEY")

if not API_KEY:
    API_KEY = input("Enter Gemini API key: ").strip()

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-flash-latest"

print("Gemini client ready.")

Gemini client ready.


In [ ]:
uploaded = files.upload()

KB_PATH = next(iter(uploaded))

with open(KB_PATH, "r", encoding="utf-8") as f:
    DATA = json.load(f)

KB = DATA["topics"]

print("Topics in knowledge base:", len(KB))

Saving kb.json to kb (3).json
Topics in knowledge base: 25


In [ ]:
print("Number of topics:", len(KB))
print("First topic fields:", list(KB[0].keys()))
print("First topic:", KB[0].get("topic"))
print("First topic keywords:", KB[0].get("keywords"))

Number of topics: 25
First topic fields: ['id', 'topic', 'category', 'keywords', 'aliases', 'supported_intents', 'definition', 'formula', 'calculation_steps', 'example', 'beginner', 'intermediate', 'advanced', 'summary']
First topic: Simple Interest
First topic keywords: ['simple interest', 'simple interest formula']


In [ ]:
class TutorResponse(BaseModel):
    topic: str
    intent: str
    difficulty: str
    definition: str
    calculation_steps: Optional[List[str]] = None
    formula: Optional[str] = None
    example: str
    summary: str

In [ ]:
def normalize(text):
    return re.sub(r"[^a-z0-9₹%/\s-]", " ", text.lower())


def parse_response(text):
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    return TutorResponse.model_validate_json(text)


def fallback_response():
    return TutorResponse(
        topic="Unsupported",
        intent="fallback",
        difficulty="beginner",
        definition="This tutor supports only predefined finance-learning topics.",
        calculation_steps=None,
        formula=None,
        example="N/A",
        summary="Please ask an educational question covered by the finance knowledge base."
    )

In [ ]:
def detect_topic(query):

    q = normalize(query)

    best_topic = None
    best_score = 0

    for topic in KB:

        score = 0

        for keyword in topic.get("keywords", []):
            keyword_normalized = normalize(keyword)

            if keyword_normalized in q:
                score += 1

        if score > best_score:
            best_score = score
            best_topic = topic

    return best_topic

In [ ]:
queries = [
    "Explain the P/E ratio",
    "What is diversification?",
    "Explain bonds",
    "What is inflation?",
    "What are dividends?"
]

for query in queries:
    topic = detect_topic(query)
    print(query, "->", topic["topic"] if topic else "No match")

Explain the P/E ratio -> P/E Ratio
What is diversification? -> Diversification
Explain bonds -> Bonds
What is inflation? -> Inflation
What are dividends? -> Dividends


In [ ]:
def detect_intent(query):
    q = normalize(query)

    if any(x in q for x in ["calculate", "compute", "formula", "find", "how much"]):
        return "calculation"

    if any(x in q for x in ["example", "real world", "scenario", "illustrate"]):
        return "example"

    if any(x in q for x in ["compare", "difference", "versus", "vs", "better than"]):
        return "comparison"

    if any(x in q for x in ["risk", "risks", "advantages", "disadvantages", "pros", "cons", "benefits"]):
        return "risk_benefit_analysis"

    if any(x in q for x in ["analyze", "analysis", "evaluate", "assess", "interpret"]):
        return "analysis"

    if any(x in q for x in ["how does", "how do", "how can", "process", "steps", "works"]):
        return "process"

    if any(x in q for x in ["should i", "which is better", "choose", "decision"]):
        return "decision_support"

    if any(x in q for x in ["what is", "what are", "define", "definition", "meaning", "explain"]):
        return "concept_explanation"

    return "general"

In [ ]:
BASE_PROMPT = """
You are a finance tutor.

Answer the student's question using only the supplied knowledge base entry.

Return JSON with:
topic
intent
difficulty
definition
calculation_steps
formula
example
summary
"""

In [ ]:
def baseline_agent(query):
    topic = detect_topic(query)

    if topic is None:
        return fallback_response()

    prompt = f"""
{BASE_PROMPT}

Knowledge base entry:
{json.dumps(topic, ensure_ascii=False)}

Student question:
{query}
"""

    response = client.interactions.create(
        model=MODEL,
        input=prompt
    )

    return parse_response(response.output_text)

In [ ]:
BASE_QUERY = "Explain the P/E ratio and give me a real-world example."

baseline_result = baseline_agent(BASE_QUERY)

print(baseline_result.model_dump_json(indent=2))

{
  "topic": "P/E Ratio",
  "intent": "explanation",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}


In [ ]:
OPT1_PROMPT = """
You are a finance tutor.

Use only the supplied knowledge base entry.
Do not introduce facts, assumptions, predictions, or personalized financial advice.
Do not invent formulas, numbers, or examples.

Return only valid JSON containing:
topic, intent, difficulty, definition,
calculation_steps, formula, example, summary.
"""

In [ ]:
def prompt_optimized_agent(query):
    topic = detect_topic(query)

    if topic is None:
        return fallback_response()

    intent = detect_intent(query)

    prompt = f"""
{OPT1_PROMPT}

Detected learning intent:
{intent}

Knowledge base entry:
{json.dumps(topic, ensure_ascii=False)}

Student question:
{query}
"""

    response = client.interactions.create(
        model=MODEL,
        input=prompt
    )

    result = parse_response(response.output_text)
    result.intent = intent

    return result

In [ ]:
result = prompt_optimized_agent(BASE_QUERY)

print(result.model_dump_json(indent=2))

{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}


In [ ]:
def intent_optimized_agent(query):
    topic = detect_topic(query)

    if topic is None:
        return fallback_response()

    intent = detect_intent(query)

    prompt = f"""
You are a finance tutor.

Use only the supplied knowledge base entry.

The student's learning intent is:
{intent}

Adapt the explanation to this learning intent.

Do not add information outside the knowledge base.
Do not provide personalized financial advice.

Knowledge base:
{json.dumps(topic, ensure_ascii=False)}

Student question:
{query}

Return only valid JSON with:
topic, intent, difficulty, definition,
calculation_steps, formula, example, summary.
"""

    response = client.interactions.create(
        model=MODEL,
        input=prompt
    )

    result = parse_response(response.output_text)
    result.intent = intent

    return result

In [ ]:
result = intent_optimized_agent(
    "Calculate the P/E ratio using the given information."
)

print(result.model_dump_json(indent=2))

{
  "topic": "P/E Ratio",
  "intent": "calculation",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}


In [ ]:
DYNAMIC_PROMPTS = {
    "concept_explanation": "Explain the financial concept clearly and focus on its meaning.",
    "calculation": "Focus on the formula and calculation steps provided in the knowledge base.",
    "example": "Focus on the example and explain how it illustrates the concept.",
    "comparison": "Focus on the similarities and differences supported by the knowledge base.",
    "risk_benefit_analysis": "Focus on the risks, benefits, advantages, and limitations provided.",
    "analysis": "Focus on interpreting the information provided in the knowledge base.",
    "process": "Explain the process using only the supplied steps.",
    "decision_support": "Explain the relevant educational factors without giving personalized advice.",
    "general": "Give a concise educational explanation."
}

In [ ]:
def dynamic_prompt_agent(query):
    topic = detect_topic(query)

    if topic is None:
        return fallback_response()

    intent = detect_intent(query)
    task = DYNAMIC_PROMPTS.get(intent, DYNAMIC_PROMPTS["general"])

    prompt = f"""
You are a finance tutor.

Use only the supplied knowledge base.

Learning intent:
{intent}

Instruction:
{task}

Do not introduce external facts or personalized financial advice.

Knowledge base:
{json.dumps(topic, ensure_ascii=False)}

Student question:
{query}

Return only valid JSON with:
topic, intent, difficulty, definition,
calculation_steps, formula, example, summary.
"""

    response = client.interactions.create(
        model=MODEL,
        input=prompt
    )

    result = parse_response(response.output_text)
    result.intent = intent

    return result

In [ ]:
result = dynamic_prompt_agent(
    "Calculate the P/E ratio using the information provided."
)

print(result.model_dump_json(indent=2))

{
  "topic": "P/E Ratio",
  "intent": "calculation",
  "difficulty": "intermediate",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}


In [ ]:
AMBIGUOUS_TERMS = {
    "market": [
        "Bull Market",
        "Bear Market",
        "Primary Market",
        "Secondary Market"
    ],
    "ratio": [
        "P/E Ratio",
        "Return on Equity",
        "Debt-to-Equity Ratio",
        "Liquidity Ratio"
    ],
    "return": [
        "Investment Return",
        "Risk-Return Tradeoff"
    ],
    "risk": [
        "Financial Risk",
        "Diversification",
        "Volatility",
        "Risk-Return Tradeoff"
    ]
}


def clarification_agent(query):
    q = normalize(query)

    for term, topic_names in AMBIGUOUS_TERMS.items():

        if term in q:
            matches = []

            for topic in KB:
                if topic["topic"] in topic_names:
                    matches.append(topic)

            if len(matches) > 1:
                print("\nYour question could refer to multiple finance topics.")

                for i, topic in enumerate(matches, 1):
                    print(f"{i}. {topic['topic']}")

                while True:
                    choice = input("\nWhich topic do you mean? ").strip()

                    if choice.isdigit():
                        index = int(choice)

                        if 1 <= index <= len(matches):
                            return matches[index - 1]

                    for topic in matches:
                        if choice.lower() == topic["topic"].lower():
                            return topic

                    print("Please enter a valid topic number or topic name.")

    return None

In [ ]:
topic = clarification_agent("Explain market")

if topic:
    print("\nSelected topic:", topic["topic"])
else:
    print("\nNo clarification required.")


Your question could refer to multiple finance topics.
1. Bull Market
2. Bear Market
3. Primary Market
4. Secondary Market

Which topic do you mean? Bear Market

Selected topic: Bear Market


In [ ]:
def validate_response(response):
    missing = []

    if not response.topic.strip():
        missing.append("topic")

    if not response.definition.strip():
        missing.append("definition")

    if not response.example.strip():
        missing.append("example")

    if not response.summary.strip():
        missing.append("summary")

    if response.intent in ["formula", "calculation"] and not response.formula:
        missing.append("formula")

    if response.intent == "calculation" and not response.calculation_steps:
        missing.append("calculation_steps")

    return len(missing) == 0, missing

In [ ]:
test_query = "Calculate the P/E ratio if the share price is ₹150 and EPS is ₹10."

result = dynamic_prompt_agent(test_query)

valid, missing = validate_response(result)

print(result.model_dump_json(indent=2))
print("Valid:", valid)
print("Missing fields:", missing)

{
  "topic": "Earnings Per Share",
  "intent": "calculation",
  "difficulty": "beginner",
  "definition": "EPS measures earnings attributable to each outstanding common share under the chosen accounting definition.",
  "calculation_steps": [
    "Identify earnings attributable to common shareholders.",
    "Identify weighted-average common shares.",
    "Divide earnings by weighted-average shares."
  ],
  "formula": "Basic EPS=earnings attributable to common shareholders/weighted-average common shares",
  "example": "₹10 million earnings divided by 2 million shares gives EPS of ₹5.",
  "summary": "EPS measures earnings attributable to each common share."
}
Valid: True
Missing fields: []


In [ ]:
def self_reflection(response, topic):
    issues = []

    valid, missing = validate_response(response)

    if not valid:
        issues.extend(
            [f"Missing field: {field}" for field in missing]
        )

    response_text = json.dumps(
        response.model_dump(),
        ensure_ascii=False
    ).lower()

    knowledge_text = json.dumps(
        topic,
        ensure_ascii=False
    ).lower()

    if response.formula:
        formula = response.formula.lower()

        if formula not in knowledge_text:
            issues.append("Formula is not present in the knowledge base.")

    if "guaranteed" in response_text:
        issues.append("Potential unsupported guarantee.")

    if "predict" in response_text:
        issues.append("Potential prediction detected.")

    return issues

In [ ]:
topic = detect_topic(BASE_QUERY)
result = dynamic_prompt_agent(BASE_QUERY)

issues = self_reflection(result, topic)

print(result.model_dump_json(indent=2))
print("Reflection:", issues if issues else "No issues detected")

{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15. This illustrates the concept by dividing the market price per share (₹150) by the earnings per share (₹10) to determine the ratio.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}
Reflection: No issues detected


In [ ]:
def retry_agent(query, max_retries=2):
    topic = detect_topic(query)

    if topic is None:
        return fallback_response(), 0

    intent = detect_intent(query)

    prompt = f"""
You are a finance tutor.

Use ONLY the supplied knowledge base.

Do not invent facts, formulas, examples, predictions,
or personalized financial advice.

Learning intent:
{intent}

Knowledge base:
{json.dumps(topic, ensure_ascii=False)}

Student question:
{query}

Return valid JSON containing:
topic, intent, difficulty, definition,
calculation_steps, formula, example, summary.
"""

    for attempt in range(max_retries + 1):

        try:
            response = client.interactions.create(
                model=MODEL,
                input=prompt
            )

            result = parse_response(response.output_text)

            valid, missing = validate_response(result)

            if not valid:
                raise ValueError(f"Missing fields: {missing}")

            issues = self_reflection(result, topic)

            if issues:
                raise ValueError(f"Validation failed: {issues}")

            result.intent = intent

            return result, attempt + 1

        except Exception as e:

            if attempt == max_retries:
                return fallback_response(), attempt + 1

In [ ]:
result, attempts = retry_agent(BASE_QUERY)

print(result.model_dump_json(indent=2))
print("Attempts:", attempts)

{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}
Attempts: 1


In [ ]:
def normalize_response(data, topic, intent, difficulty):
    return TutorResponse(
        topic=data.get("topic", topic),
        intent=data.get("intent", intent),
        difficulty=data.get("difficulty", difficulty),
        definition=data.get("definition", data.get("explanation", "")),
        calculation_steps=data.get("calculation_steps"),
        formula=data.get("formula"),
        example=data.get("example", ""),
        summary=data.get("summary", "")
    )

In [ ]:
DIFFICULTY_LEVELS = {
    "beginner": "Use simple language and explain the basic idea clearly.",
    "intermediate": "Use moderate detail and explain important relationships.",
    "advanced": "Use precise financial terminology and deeper interpretation."
}

In [ ]:
def adaptive_agent(query, difficulty):

    topic = detect_topic(query)

    if not topic:
        return fallback_response()

    intent = detect_intent(query)

    level_text = topic.get(difficulty, topic["definition"])

    if difficulty == "beginner":
        definition = topic["definition"]
        summary = topic["summary"]
        example = topic["example"]

    elif difficulty == "intermediate":
        definition = level_text
        summary = (
            f"{level_text} "
            f"The relevant formula is {topic['formula']}"
        )
        example = topic["example"]

    else:
        definition = level_text
        summary = (
            f"{level_text} "
            f"Interpret the concept using the supplied formula and example, "
            f"while considering the assumptions and limitations of the measure."
        )
        example = (
            f"{topic['example']} "
            f"This example demonstrates how the concept is applied."
        )

    return TutorResponse(
        topic=topic["topic"],
        intent=intent,
        difficulty=difficulty,
        definition=definition,
        calculation_steps=(
            topic["calculation_steps"]
            if difficulty != "beginner"
            else None
        ),
        formula=(
            topic["formula"]
            if difficulty != "beginner"
            else None
        ),
        example=example,
        summary=summary
    )

In [ ]:
query = "Explain the P/E ratio and give me an example."

for level in ["beginner", "intermediate", "advanced"]:
    result = adaptive_agent(query, level)

    print(level.upper())
    print(result.model_dump_json(indent=2))

BEGINNER
{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": null,
  "formula": null,
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}
INTERMEDIATE
{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "intermediate",
  "definition": "The P/E ratio shows how many units of share price correspond to one unit of earnings per share. It is calculated as market price per share divided by EPS and is often used to compare valuation across companies or periods.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹1

In [ ]:
conversation_history = []


def resolve_context_topic(query):

    topic = detect_topic(query)

    if topic:
        return topic

    q = normalize(query)

    context_words = [
        "it",
        "its",
        "this",
        "that",
        "they",
        "their",
        "the same"
    ]

    if any(word in q.split() for word in context_words):
        if conversation_history:
            last_topic_id = conversation_history[-1]["topic_id"]

            for topic in KB:
                if topic["id"] == last_topic_id:
                    return topic

    return None

In [ ]:
def contextual_agent(query):

    topic = resolve_context_topic(query)

    if not topic:
        return fallback_response()

    intent = detect_intent(query)

    previous_context = ""

    if conversation_history:
        previous_context = "\n".join(
            f"Question: {item['query']}\n"
            f"Topic: {item['topic']}"
            for item in conversation_history[-2:]
        )

    prompt = f"""
You are a deterministic finance tutor.

Use ONLY the supplied knowledge.

Current question:
{query}

Resolved topic:
{topic["topic"]}

Detected intent:
{intent}

Previous conversation:
{previous_context if previous_context else "None"}

Knowledge base entry:
{json.dumps(topic, ensure_ascii=False)}

Answer the student's current question using only this knowledge.

Return JSON with these fields when relevant:
topic
intent
difficulty
definition
calculation_steps
formula
example
summary

Do not invent information.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    )

    data = json.loads(response.text)

    result = TutorResponse(
        topic=topic["topic"],
        intent=intent,
        difficulty="intermediate",
        definition=data.get("definition", topic.get("definition", "")),
        calculation_steps=data.get(
            "calculation_steps",
            topic.get("calculation_steps")
        ),
        formula=data.get(
            "formula",
            topic.get("formula")
        ),
        example=data.get(
            "example",
            topic.get("example", "")
        ),
        summary=data.get(
            "summary",
            topic.get("summary", "")
        )
    )

    conversation_history.append({
        "query": query,
        "topic_id": topic["id"],
        "topic": topic["topic"]
    })

    return result

In [ ]:
conversation_history.clear()

first = contextual_agent("Explain the P/E ratio.")

print("First question")
print(first.model_dump_json(indent=2))

second = contextual_agent("What is its formula?")

print("\nFollow-up question")
print(second.model_dump_json(indent=2))

First question
{
  "topic": "P/E Ratio",
  "intent": "concept_explanation",
  "difficulty": "intermediate",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}

Follow-up question
{
  "topic": "P/E Ratio",
  "intent": "calculation",
  "difficulty": "intermediate",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."


In [ ]:
def refine_response(response):
    data = response.model_dump()

    for field in [
        "topic",
        "definition",
        "example",
        "summary"
    ]:
        data[field] = re.sub(
            r"\s+",
            " ",
            data[field]
        ).strip()

    if data["formula"]:
        data["formula"] = re.sub(
            r"\s+",
            " ",
            data["formula"]
        ).strip()

    if data["calculation_steps"]:
        data["calculation_steps"] = [
            re.sub(r"\s+", " ", step).strip()
            for step in data["calculation_steps"]
            if step.strip()
        ]

    return TutorResponse(**data)

In [ ]:
result = dynamic_prompt_agent(BASE_QUERY)

refined = refine_response(result)

print(refined.model_dump_json(indent=2))

{
  "topic": "P/E Ratio",
  "intent": "example",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15. This illustrates the concept by dividing the market price per share (₹150) by the earnings per share (₹10), demonstrating that 15 units of share price correspond to one unit of earnings per share.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}


In [ ]:
def finance_tutor(query, difficulty="intermediate"):
    start = time.perf_counter()

    topic = detect_topic(query)

    if topic is None:
        return {
            "response": fallback_response(),
            "intent": "fallback",
            "difficulty": difficulty,
            "attempts": 0,
            "latency": round(time.perf_counter() - start, 3)
        }

    intent = detect_intent(query)

    history = conversation_history[-2:]

    task = DYNAMIC_PROMPTS.get(
        intent,
        DYNAMIC_PROMPTS["general"]
    )

    prompt = f"""
You are a deterministic finance tutoring assistant.

Use ONLY the supplied knowledge base and conversation context.

Do not add external facts.
Do not invent information.
Do not make predictions.
Do not provide personalized financial advice.

Student level:
{difficulty}

Learning intent:
{intent}

Task:
{task}

Previous conversation:
{json.dumps(history, ensure_ascii=False)}

Knowledge base:
{json.dumps(topic, ensure_ascii=False)}

Question:
{query}

Return only valid JSON containing:
topic
intent
difficulty
definition
calculation_steps
formula
example
summary
"""

    result = None
    attempts = 0

    for attempt in range(3):

        try:
            response = client.interactions.create(
                model=MODEL,
                input=prompt
            )

            candidate = parse_response(response.output_text)

            valid, missing = validate_response(candidate)

            if not valid:
                raise ValueError(f"Missing fields: {missing}")

            issues = self_reflection(candidate, topic)

            if issues:
                raise ValueError(f"Validation failed: {issues}")

            candidate.intent = intent
            candidate.difficulty = difficulty

            result = refine_response(candidate)

            attempts = attempt + 1
            break

        except Exception:
            attempts = attempt + 1

    if result is None:
        result = fallback_response()

    conversation_history.append({
        "query": query,
        "response": result.model_dump()
    })

    conversation_history[:] = conversation_history[-2:]
    latency = round(time.perf_counter() - start, 3)

    log_interaction(
        question=query,
        intent=intent,
        response_time=latency,
        response=result
    )

    return {
      "response": result,
      "intent": intent,
      "difficulty": difficulty,
      "attempts": attempts,
      "latency": latency
    }

In [ ]:
query = "Explain liquidity ratio."

for level in ["beginner", "intermediate", "advanced"]:
    conversation_history.clear()

    print(f"DIFFICULTY: {level.upper()}")

    result = finance_tutor(
        query,
        difficulty=level
    )

    print(result["response"].model_dump_json(indent=2))
    print("Intent:", result["intent"])
    print("Difficulty:", result["difficulty"])
    print("Attempts:", result["attempts"])
    print("Latency:", result["latency"], "seconds")

DIFFICULTY: BEGINNER
{
  "topic": "Liquidity Ratio",
  "intent": "concept_explanation",
  "difficulty": "beginner",
  "definition": "A liquidity ratio measures ability to meet short-term obligations using relevant short-term resources.",
  "calculation_steps": [
    "Identify current assets.",
    "Identify current liabilities.",
    "Divide current assets by current liabilities."
  ],
  "formula": "Current ratio=current assets/current liabilities",
  "example": "₹200 lakh current assets and ₹100 lakh current liabilities give a current ratio of 2.0.",
  "summary": "Liquidity ratios assess a company's ability to meet short-term obligations."
}
Intent: concept_explanation
Difficulty: beginner
Attempts: 1
Latency: 12.54 seconds
DIFFICULTY: INTERMEDIATE
{
  "topic": "Liquidity Ratio",
  "intent": "concept_explanation",
  "difficulty": "intermediate",
  "definition": "A liquidity ratio measures ability to meet short-term obligations using relevant short-term resources.",
  "calculation_step

In [ ]:
query = "Explain the P/E ratio."
for level in ["beginner", "intermediate", "advanced"]:
    conversation_history.clear()

    print(f"DIFFICULTY: {level.upper()}")

    result = finance_tutor(
        query,
        difficulty=level
    )

    print(result["response"].model_dump_json(indent=2))
    print("Intent:", result["intent"])
    print("Difficulty:", result["difficulty"])
    print("Attempts:", result["attempts"])
    print("Latency:", result["latency"], "seconds")

DIFFICULTY: BEGINNER
{
  "topic": "P/E Ratio",
  "intent": "concept_explanation",
  "difficulty": "beginner",
  "definition": "The price-to-earnings ratio compares share price with earnings per share.",
  "calculation_steps": [
    "Identify the market price per share.",
    "Identify earnings per share.",
    "Divide market price per share by EPS.",
    "Interpret the resulting multiple in context."
  ],
  "formula": "P/E=market price per share/EPS",
  "example": "A ₹150 share with EPS ₹10 has a P/E of 15.",
  "summary": "The P/E ratio compares market price per share with earnings per share."
}
Intent: concept_explanation
Difficulty: beginner
Attempts: 1
Latency: 5.433 seconds
DIFFICULTY: INTERMEDIATE
{
  "topic": "P/E Ratio",
  "intent": "concept_explanation",
  "difficulty": "intermediate",
  "definition": "The P/E ratio shows how many units of share price correspond to one unit of earnings per share. It is calculated as market price per share divided by EPS and is often used to com

In [ ]:
import sqlite3
from datetime import datetime

In [ ]:
conn = sqlite3.connect("finance_tutor_logs.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS interactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    student_question TEXT,
    detected_intent TEXT,
    response_time REAL,
    response_returned TEXT
)
""")

conn.commit()

In [ ]:
def log_interaction(
    question,
    intent,
    response_time,
    response
):
    conn = sqlite3.connect("finance_tutor_logs.db")

    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO interactions (
        timestamp,
        student_question,
        detected_intent,
        response_time,
        response_returned
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        datetime.now().isoformat(),
        question,
        intent,
        response_time,
        response.model_dump_json()
    ))

    conn.commit()
    conn.close()

In [ ]:
conn = sqlite3.connect("finance_tutor_logs.db")

cursor = conn.cursor()

cursor.execute("""
SELECT * FROM interactions
ORDER BY id DESC
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

(3, '2026-08-13T04:37:22.247033', 'Explain the P/E ratio.', 'concept_explanation', 12.041, '{"topic":"Unsupported","intent":"fallback","difficulty":"beginner","definition":"This tutor supports only predefined finance-learning topics.","calculation_steps":null,"formula":null,"example":"N/A","summary":"Please ask an educational question covered by the finance knowledge base."}')
(2, '2026-08-13T04:37:10.199338', 'Explain the P/E ratio.', 'concept_explanation', 9.801, '{"topic":"P/E Ratio","intent":"concept_explanation","difficulty":"intermediate","definition":"The P/E ratio shows how many units of share price correspond to one unit of earnings per share. It is calculated as market price per share divided by EPS and is often used to compare valuation across companies or periods.","calculation_steps":["Identify the market price per share.","Identify earnings per share.","Divide market price per share by EPS.","Interpret the resulting multiple in context."],"formula":"P/E=market price per s